# Deep Hedging of Forward Starting Put: 6 Recurrence Modes (TensorFlow Backend)
This notebook trains neural network deep hedging agents for a forward starting put option ($H = - \max(S_{t_{reset}} - S_T, 0)$) where the strike is determined at step $t_{reset} = 10$.
Evaluates 6 recurrence architectures:
1. `fwd_none`: Feedforward baseline (non-recurrent)
2. `fwd_all_modes`: All memory nodes active (classic=2, aggregate=2, past_repr=2, event=2)
3. `fwd_aggr_2`: Two aggregate exponential memory nodes (aggregate=2)
4. `fwd_aggr_5bnd`: Five bounded aggregate nodes (aggregate=5, bound_aggr_states=True)
5. `fwd_aggr_5`: Five unbounded aggregate nodes (aggregate=5, bound_aggr_states=False)
6. `fwd_past_repr`: Two past representation memory nodes (past_repr=2)

**Backend**: TensorFlow 2 / Keras.
**Self-contained**: Uses local `dh_tf` module.


In [1]:
import os, sys
from pathlib import Path

NB_DIR = Path.cwd()
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import tensorflow as tf
from cdxbasics.config import Config
from IPython.display import display, Markdown

from dh_tf.world import SimpleWorld_Spot_ATM
from dh_tf.gym import VanillaDeepHedgingGym
from dh_tf.trainer import train
from configs.catalog import get_config

MODE = os.environ.get("DH_MODE", "original")
def get_epochs(default_ep):
    if os.environ.get("DH_EPOCHS"):
        return int(os.environ.get("DH_EPOCHS"))
    return 2 if MODE == "smoke" else default_ep

def get_samples(default_samples):
    return min(default_samples, 500) if MODE == "smoke" else default_samples

print(f"trainer-recurrent-fwdstart.ipynb (TF) initialized | Mode: {MODE}")


Tensorflow version 2.15.1 running on 1 CPUs and 0 GPUs


trainer-recurrent-fwdstart.ipynb (TF) initialized | Mode: smoke


## Forward Starting Put Payoff and World Setup


In [2]:
iReset = 10
def fwd_start(spots):
    terminal = np.maximum(1.0 * spots[:, iReset] - spots[:, -1], 0.0)
    return -terminal

fwd_start.name = f"Forward Starting Put (reset={iReset})"

cfg_ref = get_config("fwd_none", mode=MODE)
n_samples = get_samples(cfg_ref["samples"])
n_epochs = get_epochs(cfg_ref["epochs"])
batch_size = min(cfg_ref["batch_size"], n_samples)

config = Config()
config.world.samples = n_samples
config.world.steps = cfg_ref["steps"]
config.world.dt = cfg_ref["dt"]
config.world.drift = cfg_ref["drift"]
config.world.cost_s = cfg_ref["cost_s"]
config.world.black_scholes = True
config.world.payoff = fwd_start

config.gym.objective.utility = cfg_ref["utility"]
config.gym.objective.lmbda = cfg_ref["lmbda"]
config.gym.agent.network.depth = cfg_ref["depth"]
config.gym.agent.network.width = cfg_ref["width"]
config.gym.agent.network.activation = cfg_ref["activation"]

config.trainer.train.optimizer.name = "adam"
config.trainer.train.optimizer.learning_rate = cfg_ref["lr"]
config.trainer.train.optimizer.clipvalue = cfg_ref["clipvalue"]
config.trainer.train.optimizer.global_clipnorm = cfg_ref["global_clipnorm"]
config.trainer.train.batch_size = batch_size
config.trainer.train.epochs = n_epochs
config.trainer.caching.mode = "off"
config.trainer.output_level = "all" if MODE == "original" else "quiet"

print(f"Simulating Forward Start World: {n_samples} samples, {cfg_ref['steps']} steps...")
world = SimpleWorld_Spot_ATM(config.world)
val_world = world.clone(samples=max(n_samples // 2, 50))
config_base = config.copy()


Simulating Forward Start World: 500 samples, 20 steps...


## 1. `fwd_none`: Non-Recurrent Feedforward


In [3]:
c = config_base.copy()
c.gym.agent.recurrence.states.classic = 0
c.gym.agent.recurrence.states.aggregate = 0
c.gym.agent.recurrence.states.past_repr = 0
c.gym.agent.recurrence.states.event = 0
c.world.mark_done()
print(f"Training fwd_none for {n_epochs} epochs...")
gym = VanillaDeepHedgingGym(c.gym)
train(gym=gym, world=world, val_world=val_world, config=c.trainer)
print("Finished fwd_none.")


Training fwd_none for 2 epochs...



Finished fwd_none.


## 2. `fwd_all_modes`: All Recurrence Nodes (classic=2, aggregate=2, past_repr=2, event=2)


In [5]:
c = config_base.copy()
c.gym.agent.recurrence.states.classic = 2
c.gym.agent.recurrence.states.aggregate = 2
c.gym.agent.recurrence.states.past_repr = 2
c.gym.agent.recurrence.states.event = 2
c.world.mark_done()
print(f"Training fwd_all_modes for {n_epochs} epochs...")
gym = VanillaDeepHedgingGym(c.gym)
train(gym=gym, world=world, val_world=val_world, config=c.trainer)
print("Finished fwd_all_modes.")


Training fwd_all_modes for 2 epochs...


Finished fwd_all_modes.


## 3. `fwd_aggr_2`: Aggregate Nodes Only (aggregate=2)


In [7]:
c = config_base.copy()
c.gym.agent.recurrence.states.classic = 0
c.gym.agent.recurrence.states.aggregate = 2
c.gym.agent.recurrence.states.past_repr = 0
c.gym.agent.recurrence.states.event = 0
c.world.mark_done()
print(f"Training fwd_aggr_2 for {n_epochs} epochs...")
gym = VanillaDeepHedgingGym(c.gym)
train(gym=gym, world=world, val_world=val_world, config=c.trainer)
print("Finished fwd_aggr_2.")


Training fwd_aggr_2 for 2 epochs...


Finished fwd_aggr_2.


## 4. `fwd_aggr_5bnd`: Bounded Aggregate Nodes (aggregate=5, bound_aggr_states=True)


In [9]:
c = config_base.copy()
c.gym.agent.recurrence.states.classic = 0
c.gym.agent.recurrence.states.aggregate = 5
c.gym.agent.recurrence.bound_aggr_states = True
c.gym.agent.recurrence.states.past_repr = 0
c.gym.agent.recurrence.states.event = 0
c.world.mark_done()
print(f"Training fwd_aggr_5bnd for {n_epochs} epochs...")
gym = VanillaDeepHedgingGym(c.gym)
train(gym=gym, world=world, val_world=val_world, config=c.trainer)
print("Finished fwd_aggr_5bnd.")


Training fwd_aggr_5bnd for 2 epochs...


Finished fwd_aggr_5bnd.


## 5. `fwd_aggr_5`: Unbounded Aggregate Nodes (aggregate=5, bound_aggr_states=False)


In [11]:
c = config_base.copy()
c.gym.agent.recurrence.states.classic = 0
c.gym.agent.recurrence.states.aggregate = 5
c.gym.agent.recurrence.bound_aggr_states = False
c.gym.agent.recurrence.states.past_repr = 0
c.gym.agent.recurrence.states.event = 0
c.world.mark_done()
print(f"Training fwd_aggr_5 for {n_epochs} epochs...")
gym = VanillaDeepHedgingGym(c.gym)
train(gym=gym, world=world, val_world=val_world, config=c.trainer)
print("Finished fwd_aggr_5.")


Training fwd_aggr_5 for 2 epochs...


Finished fwd_aggr_5.


## 6. `fwd_past_repr`: Past Representation Nodes (past_repr=2)


In [13]:
c = config_base.copy()
c.gym.agent.recurrence.states.classic = 0
c.gym.agent.recurrence.states.aggregate = 0
c.gym.agent.recurrence.states.past_repr = 2
c.gym.agent.recurrence.states.event = 0
c.world.mark_done()
print(f"Training fwd_past_repr for {n_epochs} epochs...")
gym = VanillaDeepHedgingGym(c.gym)
train(gym=gym, world=world, val_world=val_world, config=c.trainer)
print("Finished fwd_past_repr.")


Training fwd_past_repr for 2 epochs...


Finished fwd_past_repr.


### Summary
Completed training for all 6 forward start put recurrence variants under TensorFlow backend.
